In [5]:
import pandas as pd
import numpy as np
import utils.fetch_dbs

In [6]:
datasets = pd.read_csv('HDCA_v2_datasets.csv')
datasets

,title,doi,accession,institute,assay,key,avialability,status,comment,simone comment,link
0,Calvanese_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04571-x,GSE162950,Broad_Stem_Cell_Research_centre,NaN,NaN,open,reprocessed by Alex,"we use just four samples, I guess AGM stands f...",NaN,NaN
1,Dong_et_al_2022_CancerCell,https://doi.org/10.1016/j.ccell.2020.08.014,GSE137804,Children's Hospital Affiliated to Fudan Univer...,NaN,NaN,open,NaN,"just four samples, to be selected by names",NaN,NaN
2,Braun_et_al._2022_bioRxiv,https://www.science.org/doi/10.1126/science.ad...,EGAD50000001295,Karolinska,NaN,NaN,EGA,reprocessed by Alex,EGAS00001004107,NaN,NaN
3,Braun_et_al._2022_bioRxiv,https://www.science.org/doi/10.1126/science.ad...,EGAD00001006049,Karolinska,NaN,NaN,EGA,reprocessed by Alex,EGAS00001004107,NaN,NaN
4,Colin_et_al_2022_ScienceDirect,https://doi.org/10.1016/j.jtos.2021.03.010,GSE155683,Newcastle_University,NaN,NaN,open,NaN,NaN,NaN,NaN
5,Yu_et_al_2020_Cell,https://doi.org/10.1016/j.cell.2021.04.028,E-MTAB-10187,Roche,NaN,NaN,open,reprocessed by Alex,NaN,NaN,NaN
6,Miller_et_al_2020_Developmental_Cell,https://doi.org/10.1016/j.devcel.2020.01.033,E-MTAB-8221,University_of_Michigan,NaN,NaN,open,reprocessed by Alex,NaN,NaN,NaN
7,Sridhar_et_al_2020_CellPress,https://doi.org/10.1016/j.celrep.2020.01.007,GSE142526,University_of_Washington,NaN,NaN,open,NaN,NaN,NaN,NaN
8,Garcia_Alonso_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04918-4,E-MTAB-10551,Wellcome_Sanger,10x RNA-seq,NaN,open,reprocessed by Alex,NaN,NaN,NaN
9,Garcia_Alonso_et_al_2022_nature,https://doi.org/10.1038/s41586-022-04918-4,E-MTAB-11708,Wellcome_Sanger,10x multiome,NaN,open,NaN,not sure it was used,NaN,NaN


# Collect metadata from databases
## ArrayExpress

In [3]:
inx = datasets.accession.str.startswith('E-MTAB')
inx = inx[inx].index
for i in inx:
    a = datasets.accession[i]
    print(a)
    url = 'https://www.ebi.ac.uk/biostudies/files/'+a+'/'+a+'.sdrf.txt'
    if pd.notna(datasets.key[i]):
        url = url+"?key="+datasets.key[i]
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

5
6
8
9
10
13
14
16
17
18
19
20
22
23
29
30
31
32
33


## EGA

In [31]:
f = datasets.accession.str.startswith('EGAD')
for a in datasets.accession[f]:
    print(a)
    d = utils.fetch_dbs.get_EGAD_files(a)
    d.to_csv("datasets_metadata/"+a+".csv")

EGAD50000001295
EGAD00001006049
EGAD00001009801
EGAD00001015384


## GEO

In [3]:
f = datasets.accession.str.startswith('GSE')
for a in datasets.accession[f]:
    print(a)
    url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/'+a[:6]+'nnn/'+a+'/soft/'+a+'_family.soft.gz'
    d = utils.fetch_dbs.get_geo_family_soft_samples(url)
    d.to_csv("datasets_metadata/"+a+".csv")

GSE162950
GSE137804
GSE155683
GSE142526
GSE260715
GSE157329
GSE264407
GSE271304


## ENA

In [4]:
f = datasets.accession.str.startswith('PRJEB')
for a in datasets.accession[f]:
    print(a)
    url = 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession='+a+'&result=read_run&fields=study_accession,experiment_accession,sample_accession,secondary_sample_accession,sample_title&format=tsv'
    d = pd.read_csv(url, sep="\t")
    d.to_csv("datasets_metadata/"+a+".csv")

PRJEB37165
PRJEB77091


## CNCB
didn't find working API, anyway we need to get access first

# Concatenate metadata
The goal is to get list of samples with minimal info:

1. db_acc
2. sample_acc
3. author_sample_id
4. [library_type]
5. [description]